In [1]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm import tqdm
import gc

# CONFIG
INPUT_CSV = '/kaggle/input/datasets/dhatrisriram/sc-judgements/sc_judgments_extracted.csv'  # Update this!

# Path to your trained model (upload results.zip and extract)
MODEL_DIR = '/kaggle/input/datasets/dhatrisriram/best-model/best_model/'  # Update this!

# Output directory
OUTPUT_DIR = '/kaggle/working/sc_predictions'

CHUNK_SIZE  = 200_000
BATCH_SIZE  = 256   # 🔥 BIG speed boost
MAX_LEN     = 128
NUM_WORKERS = 4

Path(OUTPUT_DIR).mkdir(exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_FP16 = device.type == 'cuda'

print(f"Device: {device}, FP16: {USE_FP16}")

Device: cuda, FP16: True


In [2]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

if USE_FP16:
    model = model.half()

model.eval()

id2label = {
    0: "Arguments of Petitioner",
    1: "Arguments of Respondent",
    2: "Decision",
    3: "Facts",
    4: "Issue",
    5: "Reasoning"
}

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [3]:
class LegalDataset(Dataset):
    def __init__(self, texts):
        self.texts = texts

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx]


def collate_fn(batch_texts):
    return tokenizer(
        batch_texts,
        truncation=True,
        padding=True,
        max_length=MAX_LEN,
        return_tensors='pt'
    )

In [4]:
reader = pd.read_csv(INPUT_CSV, chunksize=CHUNK_SIZE)

output_files = []

for chunk_idx, chunk_df in enumerate(reader):
    
    print(f"\n🚀 Processing Chunk {chunk_idx+1} ({len(chunk_df):,} rows)")
    
    # CLEAN
    chunk_df = chunk_df[chunk_df['Text'].notna()]
    chunk_df = chunk_df[chunk_df['Text'].astype(str).str.strip() != '']
    chunk_df = chunk_df.reset_index(drop=True)

    dataset = LegalDataset(chunk_df['Text'].tolist())

    dataloader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        collate_fn=collate_fn,
        pin_memory=True
    )

    preds_all = []
    confs_all = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"Chunk {chunk_idx+1}"):

            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}

            if USE_FP16:
                with torch.cuda.amp.autocast():
                    logits = model(**batch).logits
            else:
                logits = model(**batch).logits

            probs = torch.softmax(logits.float(), dim=-1)

            preds = torch.argmax(probs, dim=1).cpu().numpy()
            confs = probs.max(dim=1).values.cpu().numpy()

            preds_all.extend(preds)
            confs_all.extend(confs)

    # SAVE IMMEDIATELY (VERY IMPORTANT)
    chunk_df['predicted_label'] = [id2label[p] for p in preds_all]
    chunk_df['confidence'] = confs_all

    out_file = f"{OUTPUT_DIR}/chunk_{chunk_idx}.csv"
    chunk_df.to_csv(out_file, index=False)
    output_files.append(out_file)

    print(f"✅ Saved: {out_file}")

    # FREE MEMORY
    del dataset, dataloader, preds_all, confs_all
    gc.collect()
    torch.cuda.empty_cache()


🚀 Processing Chunk 1 (200,000 rows)


Chunk 1:   0%|          | 0/782 [00:00<?, ?it/s]/tmp/ipykernel_55/3966908424.py:34: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Chunk 1: 100%|██████████| 782/782 [05:34<00:00,  2.34it/s]


✅ Saved: /kaggle/working/sc_predictions/chunk_0.csv

🚀 Processing Chunk 2 (200,000 rows)


Chunk 2:   0%|          | 0/782 [00:00<?, ?it/s]/tmp/ipykernel_55/3966908424.py:34: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Chunk 2: 100%|██████████| 782/782 [05:46<00:00,  2.26it/s]


✅ Saved: /kaggle/working/sc_predictions/chunk_1.csv

🚀 Processing Chunk 3 (200,000 rows)


...

✅ Saved: /kaggle/working/sc_predictions/chunk_24.csv

🚀 Processing Chunk 26 (200,000 rows)


Chunk 26:   0%|          | 0/782 [00:00<?, ?it/s]/tmp/ipykernel_55/3966908424.py:34: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Chunk 26: 100%|██████████| 782/782 [05:44<00:00,  2.27it/s]


✅ Saved: /kaggle/working/sc_predictions/chunk_25.csv

🚀 Processing Chunk 27 (53,679 rows)


Chunk 27:   0%|          | 0/210 [00:00<?, ?it/s]/tmp/ipykernel_55/3966908424.py:34: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Chunk 27: 100%|██████████| 210/210 [01:32<00:00,  2.28it/s]


✅ Saved: /kaggle/working/sc_predictions/chunk_26.csv


In [5]:
print("🔗 Merging all chunks...")

dfs = [pd.read_csv(f) for f in output_files]
final_df = pd.concat(dfs, ignore_index=True)

final_path = f"{OUTPUT_DIR}/final_predictions.csv"
final_df.to_csv(final_path, index=False)

print(f"✅ Final saved: {final_path}")

🔗 Merging all chunks...
✅ Final saved: /kaggle/working/sc_predictions/final_predictions.csv
